<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/05_SARIMA_Modeling_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 5: МОДЕЛИ ВРЕМЕННЫХ РЯДОВ (SARIMA)
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"Период: с {df.index.min()} по {df.index.max()}")

# ============================================================
# 3. МЕТОДОЛОГИЧЕСКОЕ ОБОСНОВАНИЕ ВЫБОРА SARIMA
# ============================================================

print("\n" + "="*60)
print("3. ПОЧЕМУ SARIMA, А НЕ PROPHET")
print("="*60)

print("""
📌 КЛЮЧЕВАЯ ПРОБЛЕМА: АВТОКОРРЕЛЯЦИЯ ОСТАТКОВ 4-ГО ПОРЯДКА

Во всех предыдущих моделях (базовая Ridge, полная модель с 38 признаками)
диагностика остатков на обучающей выборке выявила значимую автокорреляцию
4-го порядка (тест Бройша-Годфри, p < 0.05).

Это означает, что ошибки модели коррелируют с ошибками 4 месяца назад,
что указывает на неучтенную квартальную (сезонную) структуру.

ПОЧЕМУ НЕ PROPHET:
- Prophet моделирует сезонность через ряды Фурье
- Не учитывает автокорреляцию в остатках напрямую
- При автокоррелированных ошибках дает смещенные прогнозы

ПОЧЕМУ SARIMA:
- Явно моделирует автокорреляцию через AR/MA компоненты
- Сезонные компоненты (P,D,Q,s) могут учесть квартальную структуру
- Ljung-Box тест позволяет проверить, устранена ли автокорреляция

ВЫВОД: Выбираем SARIMA для решения проблемы автокорреляции.
""")

# ============================================================
# 4. АНАЛИЗ СТАЦИОНАРНОСТИ
# ============================================================

print("\n" + "="*60)
print("4. АНАЛИЗ СТАЦИОНАРНОСТИ")
print("="*60)

# 4.1. Тест ADF
print("\n🔍 Тест Дики-Фуллера (ADF):")
adf_result = adfuller(df['DEPOS'], autolag='AIC')
print(f"   ADF-статистика: {adf_result[0]:.4f}")
print(f"   p-значение: {adf_result[1]:.4f}")
print(f"   Вывод: {'Стационарный ✅' if adf_result[1] < 0.05 else 'Нестационарный ❌'}")

# 4.2. Тест KPSS
print("\n🔍 Тест KPSS:")
kpss_result = kpss(df['DEPOS'], regression='ct')
print(f"   KPSS-статистика: {kpss_result[0]:.4f}")
print(f"   p-значение: {kpss_result[1]:.4f}")
print(f"   Вывод: {'Стационарный ✅' if kpss_result[1] > 0.05 else 'Нестационарный ❌'}")

# 4.3. График ряда
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['DEPOS'], linewidth=2, color='steelblue')
plt.title('Объем вкладов населения РФ (DEPOS)', fontsize=14)
plt.xlabel('Дата')
plt.ylabel('млрд руб.')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_depos_series.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)
# ============================================================

print("\n" + "="*60)
print("5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)")
print("="*60)

# Первое дифференцирование
df['DEPOS_diff1'] = df['DEPOS'].diff()

# ADF тест для diff1
adf_diff1 = adfuller(df['DEPOS_diff1'].dropna(), autolag='AIC')
print(f"\n📊 ADF тест после d=1:")
print(f"   ADF = {adf_diff1[0]:.4f}")
print(f"   p = {adf_diff1[1]:.4f}")
print(f"   → {'Ряд стационарен, d=1 достаточно ✅' if adf_diff1[1] < 0.05 else 'Требуется d=2 ❌'}")

# График после первого дифференцирования
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df.index, df['DEPOS_diff1'], linewidth=1.5, color='steelblue')
axes[0].set_title('DEPOS после первого дифференцирования (d=1)')
axes[0].set_ylabel('Δ DEPOS')
axes[0].grid(True, alpha=0.3)

axes[1].text(0.5, 0.5,
             f'ADF тест для d=1:\nADF = {adf_diff1[0]:.4f}\np = {adf_diff1[1]:.4f}\nВывод: {"Стационарный ✅" if adf_diff1[1] < 0.05 else "Нестационарный ❌"}',
             transform=axes[1].transAxes, fontsize=12, ha='center', va='center',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
axes[1].axis('off')

plt.tight_layout()
plt.savefig('05_diff1_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

d_order = 1  # Первое дифференцирование достаточно

# ============================================================
# 6. АНАЛИЗ ACF И PACF ДЛЯ ПОДБОРА ПАРАМЕТРОВ
# ============================================================

print("\n" + "="*60)
print("6. АНАЛИЗ ACF И PACF")
print("="*60)

# Создаем обучающую выборку (последние 12 месяцев - тест)
train_size = len(df) - 12
depos_train = df['DEPOS'].iloc[:train_size]
depos_test = df['DEPOS'].iloc[train_size:]

print(f"\n📊 Обучающая выборка: {len(depos_train)} записей")
print(f"📊 Тестовая выборка: {len(depos_test)} записей")
print(f"📊 Период теста: {depos_test.index[0]} — {depos_test.index[-1]}")

# Дифференцированный ряд
depos_train_diff = depos_train.diff().dropna()

# Графики ACF и PACF
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ACF исходного ряда
plot_acf(depos_train, lags=24, ax=axes[0, 0])
axes[0, 0].set_title('ACF: Исходный ряд (DEPOS)')
axes[0, 0].set_xlabel('Лаг')
axes[0, 0].set_ylabel('Автокорреляция')

# PACF исходного ряда
plot_pacf(depos_train, lags=24, ax=axes[0, 1])
axes[0, 1].set_title('PACF: Исходный ряд (DEPOS)')
axes[0, 1].set_xlabel('Лаг')
axes[0, 1].set_ylabel('Частичная автокорреляция')

# ACF дифференцированного ряда
plot_acf(depos_train_diff, lags=24, ax=axes[1, 0])
axes[1, 0].set_title('ACF: Дифференцированный ряд (d=1)')
axes[1, 0].set_xlabel('Лаг')
axes[1, 0].set_ylabel('Автокорреляция')

# PACF дифференцированного ряда
plot_pacf(depos_train_diff, lags=24, ax=axes[1, 1])
axes[1, 1].set_title('PACF: Дифференцированный ряд (d=1)')
axes[1, 1].set_xlabel('Лаг')
axes[1, 1].set_ylabel('Частичная автокорреляция')

plt.tight_layout()
plt.savefig('05_acf_pacf_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📌 Наблюдения из ACF/PACF:")
print("   - Исходный ряд: медленно убывающая ACF → нестационарность")
print("   - Дифференцированный ряд: значимые пики на лагах 3-4")
print("   - Возможная сезонность: s = 4 (квартальная)")

# ============================================================
# 7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA (GRID SEARCH)
# ============================================================

print("\n" + "="*60)
print("7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA")
print("="*60)

def evaluate_sarima_model(order, seasonal_order, train_data, test_data):
    """
    Обучает SARIMA модель и оценивает качество.
    """
    try:
        model = SARIMAX(
            train_data,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

        # Прогноз на тестовый период
        forecast = results.forecast(steps=len(test_data))

        # Метрики на тестовой выборке
        r2_test = r2_score(test_data, forecast)
        rmse_test = np.sqrt(mean_squared_error(test_data, forecast))
        mae_test = mean_absolute_error(test_data, forecast)

        # AIC и BIC
        aic = results.aic
        bic = results.bic

        return {
            'model': results,
            'forecast': forecast,
            'r2_test': r2_test,
            'rmse_test': rmse_test,
            'mae_test': mae_test,
            'aic': aic,
            'bic': bic,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Определяем сетку параметров
# Для квартальной сезонности (s=4) пробуем разные варианты
p_range = range(0, 4)  # 0-3
d = 1  # уже определено
q_range = range(0, 4)  # 0-3
P_range = range(0, 2)  # 0-1
D_range = range(0, 2)  # 0-1
Q_range = range(0, 2)  # 0-1
s = 4  # квартальная сезонность

print(f"\n🔍 Запуск grid search...")
print(f"   Диапазоны: p={list(p_range)}, d={[d]}, q={list(q_range)}")
print(f"   Сезонные: P={list(P_range)}, D={list(D_range)}, Q={list(Q_range)}, s={s}")
print(f"   Всего комбинаций: {len(p_range) * len(q_range) * len(P_range) * len(D_range) * len(Q_range)}")

best_models = []

for p, q, P, D, Q in product(p_range, q_range, P_range, D_range, Q_range):
    order = (p, d, q)
    seasonal_order = (P, D, Q, s)

    result = evaluate_sarima_model(
        order, seasonal_order,
        depos_train, depos_test
    )

    if result['success']:
        best_models.append({
            'order': order,
            'seasonal_order': seasonal_order,
            'r2_test': result['r2_test'],
            'rmse_test': result['rmse_test'],
            'aic': result['aic'],
            'bic': result['bic'],
            'result': result
        })

        if len(best_models) <= 5 or result['aic'] < min(m['aic'] for m in best_models[:-1]):
            print(f"   ✅ SARIMA{order}×{seasonal_order}: AIC={result['aic']:.2f}, R²_test={result['r2_test']:.4f}")

print(f"\n📊 Grid search завершен. Найдено {len(best_models)} моделей.")

# Сортировка по AIC
best_models_sorted = sorted(best_models, key=lambda x: x['aic'])

print("\n🏆 ТОП-5 моделей по AIC:")
for i, model in enumerate(best_models_sorted[:5], 1):
    print(f"   {i}. SARIMA{model['order']}×{model['seasonal_order']}: "
          f"AIC={model['aic']:.2f}, BIC={model['bic']:.2f}, "
          f"R²_test={model['r2_test']:.4f}, RMSE={model['rmse_test']:.2f}")

# Выбираем лучшую модель по AIC
best_sarima = best_models_sorted[0]
best_order = best_sarima['order']
best_seasonal_order = best_sarima['seasonal_order']
best_result = best_sarima['result']

print(f"\n✅ Лучшая модель: SARIMA{best_order}×{best_seasonal_order}")
print(f"   AIC = {best_sarima['aic']:.2f}")
print(f"   BIC = {best_sarima['bic']:.2f}")
print(f"   R²_test = {best_sarima['r2_test']:.4f}")
print(f"   RMSE_test = {best_sarima['rmse_test']:.2f} млрд руб.")

# ============================================================
# 8. ОБУЧЕНИЕ ЛУЧШЕЙ МОДЕЛИ И ДИАГНОСТИКА
# ============================================================

print("\n" + "="*60)
print("8. ОБУЧЕНИЕ ЛУЧШЕЙ МОДЕЛИ И ДИАГНОСТИКА")
print("="*60)

# Обучаем лучшую модель с полной диагностикой
print(f"\n🔧 Обучение SARIMA{best_order}×{best_seasonal_order}...")

sarima_model = SARIMAX(
    depos_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

print("✅ Модель обучена")

# 8.1. Сводка модели
print("\n📊 Сводка модели:")
print(sarima_results.summary())

# 8.2. Диагностика остатков
print("\n🔍 Диагностика остатков SARIMA:")

# Остатки
residuals_sarima = sarima_results.resid

# Ljung-Box тест (проверка автокорреляции)
lb_test = acorr_ljungbox(residuals_sarima, lags=[4, 8, 12], return_df=True)
print("\n📊 Ljung-Box тест (автокорреляция):")
print(lb_test.round(4))
print(f"   Вывод: {'✅ Нет автокорреляции' if lb_test['lb_pvalue'].min() > 0.05 else '⚠️ Автокорреляция осталась'}")

# Графики диагностики
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Диагностика SARIMA{best_order}×{best_seasonal_order}', fontsize=14)

# График остатков
axes[0, 0].plot(residuals_sarima.index, residuals_sarima, linewidth=1)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_title('Остатки модели')
axes[0, 0].set_xlabel('Дата')
axes[0, 0].set_ylabel('Остатки')
axes[0, 0].grid(True, alpha=0.3)

# Гистограмма остатков
axes[0, 1].hist(residuals_sarima, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Распределение остатков')
axes[0, 1].set_xlabel('Остатки')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].grid(True, alpha=0.3)

# ACF остатков
plot_acf(residuals_sarima, lags=24, ax=axes[1, 0])
axes[1, 0].set_title('ACF остатков')
axes[1, 0].set_xlabel('Лаг')
axes[1, 0].set_ylabel('Автокорреляция')

# Q-Q plot
from scipy import stats
stats.probplot(residuals_sarima, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot')

plt.tight_layout()
plt.savefig('05_sarima_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Диагностика завершена")

# ============================================================
# 9. ПРОГНОЗ И СРАВНЕНИЕ
# ============================================================

print("\n" + "="*60)
print("9. ПРОГНОЗ И СРАВНЕНИЕ")
print("="*60)

# 9.1. Прогноз на тестовый период
forecast_test = sarima_results.forecast(steps=len(depos_test))

# 9.2. Прогноз на обучающей выборке (для сравнения)
fitted_values = sarima_results.fittedvalues

# Метрики на обучающей выборке
r2_train_sarima = r2_score(depos_train, fitted_values)
rmse_train_sarima = np.sqrt(mean_squared_error(depos_train, fitted_values))

# Метрики на тестовой выборке
r2_test_sarima = r2_score(depos_test, forecast_test)
rmse_test_sarima = np.sqrt(mean_squared_error(depos_test, forecast_test))
mae_test_sarima = mean_absolute_error(depos_test, forecast_test)

print(f"\n📊 SARIMA{best_order}×{best_seasonal_order}:")
print(f"   ОБУЧАЮЩАЯ выборка (n={len(depos_train)}):")
print(f"     R²_train = {r2_train_sarima:.4f}")
print(f"     RMSE_train = {rmse_train_sarima:.2f} млрд руб.")
print(f"   ТЕСТОВАЯ выборка (n={len(depos_test)}):")
print(f"     R²_test = {r2_test_sarima:.4f}")
print(f"     RMSE_test = {rmse_test_sarima:.2f} млрд руб.")
print(f"     MAE_test = {mae_test_sarima:.2f} млрд руб.")

# 9.3. Сравнение с Ridge-моделью (из блока 4)
print(f"\n📊 Сравнение с Ridge-моделью (38 признаков):")
print(f"   {'Модель':<25} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12}")
print(f"   {'-'*60}")
print(f"   {'Ridge (38 признаков)':<25} {'0.9422':<10} {'566.09':<12} {'489.89':<12}")
print(f"   {f'SARIMA{best_order}×{best_seasonal_order}':<25} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

# 9.4. Визуализация прогноза
plt.figure(figsize=(14, 7))

# Полный ряд
plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)

# Прогноз на тестовый период
plt.plot(depos_test.index, forecast_test, label=f'Прогноз SARIMA{best_order}×{best_seasonal_order}',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

# Доверительный интервал (95%)
forecast_result = sarima_results.get_forecast(steps=len(depos_test))
confidence_intervals = forecast_result.conf_int(alpha=0.05)

plt.fill_between(
    depos_test.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.25, color='#ff7f0e', label='95% доверительный интервал'
)

plt.title(f'Прогноз объема вкладов — SARIMA{best_order}×{best_seasonal_order}\n' +
          f'R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f} млрд руб.',
          fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Объем вкладов, млрд руб.')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_sarima_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 10. ИТОГОВЫЙ ВЫВОД
# ============================================================

print("\n" + "="*60)
print("10. ИТОГОВЫЙ ВЫВОД")
print("="*60)

print(f"""
📌 КЛЮЧЕВЫЕ РЕЗУЛЬТАТЫ БЛОКА 5:

1. МОДЕЛЬ: SARIMA{best_order}×{best_seasonal_order}
   - Порядок: {best_order}
   - Сезонный порядок: {best_seasonal_order}
   - AIC = {best_sarima['aic']:.2f}
   - BIC = {best_sarima['bic']:.2f}

2. КАЧЕСТВО ПРОГНОЗА:
   ┌─────────────────────────────────────────────────────────┐
   │ ОБУЧАЮЩАЯ выборка (n={len(depos_train)}):               │
   │   R²_train = {r2_train_sarima:.4f}                      │
   │   RMSE_train = {rmse_train_sarima:.2f} млрд руб.       │
   ├─────────────────────────────────────────────────────────┤
   │ ТЕСТОВАЯ выборка (n={len(depos_test)}):                 │
   │   R²_test = {r2_test_sarima:.4f}                        │
   │   RMSE_test = {rmse_test_sarima:.2f} млрд руб.          │
   │   MAE_test = {mae_test_sarima:.2f} млрд руб.           │
   └─────────────────────────────────────────────────────────┘

3. ДИАГНОСТИКА ОСТАТКОВ:
   - Ljung-Box тест: {'✅ Автокорреляция устранена' if lb_test['lb_pvalue'].min() > 0.05 else '⚠️ Автокорреляция осталась'}

4. СРАВНЕНИЕ С RIDGE (БЛОК 4):
   - Ridge (38 признаков): R²_test = 0.9422, RMSE = 566.09
   - SARIMA{best_order}×{best_seasonal_order}: R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f}
   - {'✅ SARIMA лучше' if r2_test_sarima > 0.9422 else 'ℹ️ Ridge остается лучше по тесту'}

5. ВЫВОДЫ:
   - SARIMA{best_order}×{best_seasonal_order} построена для решения проблемы автокорреляции
   - {'Автокорреляция устранена (Ljung-Box p > 0.05)' if lb_test['lb_pvalue'].min() > 0.05 else 'Автокорреляция частично устранена'}
   - Сезонная компонента s=4 учитывает квартальную структуру
   - {'SARIMA превосходит Ridge' if r2_test_sarima > 0.9422 else 'Ridge (с макроэкономическими признаками) остается предпочтительнее для прогноза'}
   - Комбинированный подход (SARIMA + Ridge) может дать лучший результат

6. СЛЕДУЮЩИЕ ШАГИ:
   - [x] SARIMA модель построена
   - [ ] Комбинированная модель (SARIMA + Ridge остатки)
   - [ ] Сравнение с Prophet (для полноты)
   - [ ] Финальный прогноз на 2026-2027
""")

print("✅ БЛОК 5 ЗАВЕРШЕН")